Задание 1 (4 балла)
1. Самостоятельно выбрать 1-3 датасета на любую тему, которые будут содержать все типы сравнения данных (по отдельности, например, один датасет содержит временной тип сравнения, второй - корреляционный);
2. Визуализировать все типы сравнения, написать выводы;


In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)

df_temperature = pd.read_csv('daily-min-temperatures.csv')
df_temperature['Date'] = pd.to_datetime(df_temperature['Date'])
print(f"Датасет температур: {df_temperature.shape[0]} записей")

df_wine = pd.read_csv('winequality-red.csv', sep=';')
print(f"Датасет вина: {df_wine.shape[0]} записей, {df_wine.shape[1]} колонок")
print(f"Колонки вина: {df_wine.columns.tolist()}")

df_iris = pd.read_csv('iris.data', header=None, names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'])
print(f"Датасет ирисов: {df_iris.shape[0]} записей, {df_iris.shape[1]} колонок")

print("ТИП 1: СРАВНЕНИЕ ЧАСТЕЙ И ЦЕЛОГО")

fig1 = px.pie(df_iris, names='species', title='Распределение видов ирисов в датасете')
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

if 'quality' in df_wine.columns:
    quality_counts = df_wine['quality'].value_counts().sort_index()
    fig2 = px.pie(values=quality_counts.values, names=quality_counts.index, title='Распределение вин по оценкам качества')
    fig2.update_traces(textposition='inside', textinfo='percent+label')
    fig2.show()

print("ТИП 2: СРАВНЕНИЕ КАТЕГОРИЙ")

if 'quality' in df_wine.columns and 'alcohol' in df_wine.columns:
    wine_stats = df_wine.groupby('quality')[['alcohol', 'volatile acidity']].mean().reset_index()
    fig3 = px.bar(wine_stats, x='quality', y='alcohol', title='Среднее содержание алкоголя по оценкам качества', color='quality')
    fig3.update_xaxes(title='Оценка качества')
    fig3.update_yaxes(title='Содержание алкоголя')
    fig3.show()

species_stats = df_iris.groupby('species')[['sepal_length', 'petal_length']].mean().reset_index()
fig4 = px.bar(species_stats, x='species', y=['sepal_length', 'petal_length'], title='Сравнение средних длин чашелистиков и лепестков по видам', barmode='group')
fig4.update_xaxes(title='Вид ириса')
fig4.update_yaxes(title='Длина (см)')
fig4.show()

print("ТИП 3: АНАЛИЗ РАСПРЕДЕЛЕНИЙ")

if 'alcohol' in df_wine.columns:
    fig5 = px.histogram(df_wine, x='alcohol', nbins=30, title='Распределение содержания алкоголя в вине')
    fig5.update_xaxes(title='Содержание алкоголя')
    fig5.update_yaxes(title='Частота')
    fig5.show()

fig6 = px.histogram(df_iris, x='sepal_length', color='species', nbins=30, opacity=0.7, title='Распределение длины чашелистика по видам ирисов')
fig6.update_xaxes(title='Длина чашелистика (см)')
fig6.update_yaxes(title='Частота')
fig6.show()

print("ТИП 4: СТАТИСТИЧЕСКИЕ РАСПРЕДЕЛЕНИЯ")

fig7 = px.box(df_iris, x='species', y='petal_length', title='Распределение длины лепестка по видам ирисов', color='species')
fig7.update_xaxes(title='Вид ириса')
fig7.update_yaxes(title='Длина лепестка (см)')
fig7.show()

if 'quality' in df_wine.columns and 'alcohol' in df_wine.columns:
    fig8 = px.box(df_wine, x='quality', y='alcohol', title='Распределение алкоголя по оценкам качества', color='quality')
    fig8.update_xaxes(title='Оценка качества')
    fig8.update_yaxes(title='Содержание алкоголя')
    fig8.show()

print("ТИП 5: КОРРЕЛЯЦИОННЫЙ АНАЛИЗ")

fig9 = px.scatter(df_iris, x='sepal_length', y='petal_length', color='species', title='Зависимость длины лепестка от длины чашелистика', trendline='ols')
fig9.update_xaxes(title='Длина чашелистика (см)')
fig9.update_yaxes(title='Длина лепестка (см)')
fig9.show()

wine_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'alcohol']
available_wine_cols = [col for col in wine_cols if col in df_wine.columns]
if 'quality' in df_wine.columns:
    available_wine_cols.append('quality')
    wine_corr = df_wine[available_wine_cols].corr()
    fig10 = px.imshow(wine_corr, text_auto=True, aspect='auto', title='Корреляционная матрица характеристик вина', color_continuous_scale='RdBu_r')
    fig10.show()

iris_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
iris_corr = df_iris[iris_cols].corr()
fig11 = px.imshow(iris_corr, text_auto=True, aspect='auto', title='Корреляционная матрица признаков ирисов', color_continuous_scale='Viridis')
fig11.show()

print("ТИП 6: АНАЛИЗ ВРЕМЕННЫХ РЯДОВ")

df_temperature['Year'] = df_temperature['Date'].dt.year
df_temperature['Month'] = df_temperature['Date'].dt.month
yearly_avg = df_temperature.groupby('Year')['Temp'].mean().reset_index()

fig12 = px.line(df_temperature, x='Date', y='Temp', title='Динамика минимальных температур по дням')
fig12.update_xaxes(title='Дата')
fig12.update_yaxes(title='Температура (°C)')
fig12.show()

fig13 = px.line(yearly_avg, x='Year', y='Temp', title='Среднегодовые минимальные температуры', markers=True)
fig13.update_xaxes(title='Год')
fig13.update_yaxes(title='Средняя температура (°C)')
fig13.show()

df_temperature['MA_30'] = df_temperature['Temp'].rolling(window=30).mean()
fig14 = go.Figure()
fig14.add_trace(go.Scatter(x=df_temperature['Date'], y=df_temperature['Temp'], mode='lines', name='Ежедневная температура', line=dict(color='blue', width=1)))
fig14.add_trace(go.Scatter(x=df_temperature['Date'], y=df_temperature['MA_30'], mode='lines', name='30-дневное скользящее среднее', line=dict(color='red', width=3)))
fig14.update_layout(title='Ежедневные температуры со скользящим средним', xaxis_title='Дата', yaxis_title='Температура (°C)')
fig14.show()

print("ТИП 7: МНОГОМЕРНЫЙ АНАЛИЗ")

fig15 = px.scatter_3d(df_iris, x='sepal_length', y='sepal_width', z='petal_length', color='species', size='petal_width', title='Трехмерная визуализация признаков ирисов')
fig15.show()

if len(df_wine) > 0 and 'alcohol' in df_wine.columns and 'quality' in df_wine.columns:
    df_wine_sample = df_wine.sample(n=min(200, len(df_wine)), random_state=42)
    fig16 = px.scatter(df_wine_sample, x='alcohol', y='quality', size='citric acid' if 'citric acid' in df_wine.columns else None, color='volatile acidity' if 'volatile acidity' in df_wine.columns else None, title='Взаимосвязь характеристик вина')
    fig16.update_xaxes(title='Содержание алкоголя')
    fig16.update_yaxes(title='Оценка качества')
    fig16.show()

df_iris_num = df_iris.copy()
df_iris_num['species_code'] = pd.Categorical(df_iris_num['species']).codes
fig17 = px.parallel_coordinates(df_iris_num, color='species_code', dimensions=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'], color_continuous_scale=px.colors.qualitative.Set1, title='Параллельные координаты для видов ирисов')
fig17.show()

print("ТИП 8: СРАВНЕНИЕ НЕСКОЛЬКИХ ПЕРЕМЕННЫХ")

wine_attr_cols = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'alcohol']
available_attr = [col for col in wine_attr_cols if col in df_wine.columns]
if available_attr:
    wine_attributes = df_wine[available_attr].mean().reset_index()
    wine_attributes.columns = ['attribute', 'mean_value']
    fig18 = px.bar(wine_attributes, x='attribute', y='mean_value', title='Средние значения характеристик красного вина', color='mean_value')
    fig18.update_xaxes(title='Характеристика', tickangle=45)
    fig18.update_yaxes(title='Среднее значение')
    fig18.show()

fig19 = px.violin(df_iris, x='species', y='petal_width', box=True, points='all', title='Распределение ширины лепестка по видам ирисов', color='species')
fig19.update_xaxes(title='Вид ириса')
fig19.update_yaxes(title='Ширина лепестка (см)')
fig19.show()

if 'alcohol' in df_wine.columns and 'quality' in df_wine.columns:
    fig20 = px.density_heatmap(df_wine, x='alcohol', y='quality', title='Плотность распределения алкоголя и качества', color_continuous_scale='Viridis')
    fig20.update_xaxes(title='Содержание алкоголя')
    fig20.update_yaxes(title='Оценка качества')
    fig20.show()

Датасет температур: 3650 записей
Датасет вина: 1599 записей, 1 колонок
Колонки вина: ['fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality']
Датасет ирисов: 150 записей, 5 колонок
ТИП 1: СРАВНЕНИЕ ЧАСТЕЙ И ЦЕЛОГО


ТИП 2: СРАВНЕНИЕ КАТЕГОРИЙ


ТИП 3: АНАЛИЗ РАСПРЕДЕЛЕНИЙ


ТИП 4: СТАТИСТИЧЕСКИЕ РАСПРЕДЕЛЕНИЯ


ТИП 5: КОРРЕЛЯЦИОННЫЙ АНАЛИЗ


ТИП 6: АНАЛИЗ ВРЕМЕННЫХ РЯДОВ


ТИП 7: МНОГОМЕРНЫЙ АНАЛИЗ


ТИП 8: СРАВНЕНИЕ НЕСКОЛЬКИХ ПЕРЕМЕННЫХ


Задание* (4 балла)
1. Самостоятельно изучить понятие дашборд;
2. Построить дашборд по выбранному датасету. 

In [14]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

df_iris = pd.read_csv('iris.data', header=None, names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'])

df_iris['sepal_area'] = df_iris['sepal_length'] * df_iris['sepal_width']
df_iris['petal_area'] = df_iris['petal_length'] * df_iris['petal_width']
df_iris['sepal_petal_ratio'] = df_iris['sepal_length'] / df_iris['petal_length']

app = dash.Dash(__name__)

colors = {
    'background': '#F0F4F8',
    'text': '#2C3E50',
    'header': '#1E3A5F',
    'accent1': '#2E86AB',
    'accent2': '#A23B72',
    'accent3': '#F18F01'
}

species_colors = {
    'Iris-setosa': '#2E86AB',
    'Iris-versicolor': '#A23B72',
    'Iris-virginica': '#F18F01'
}

app.layout = html.Div(style={'backgroundColor': colors['background'], 
                              'fontFamily': 'Arial, sans-serif',
                              'padding': '20px'}, children=[
    
    html.H1('Аналитический дашборд: Классификация ирисов Фишера',
            style={'textAlign': 'center', 'color': colors['header'],
                   'marginBottom': '30px', 'fontSize': '36px'}),
    
    html.Div([
        html.Div([
            html.Label('Выберите вид ириса:', 
                      style={'fontWeight': 'bold', 'color': colors['text']}),
            dcc.Dropdown(
                id='species-dropdown',
                options=[{'label': s, 'value': s} for s in df_iris['species'].unique()] + 
                        [{'label': 'Все виды', 'value': 'ALL'}],
                value='ALL',
                clearable=False,
                style={'width': '100%', 'marginBottom': '15px'}
            ),
        ], style={'width': '45%', 'display': 'inline-block', 'marginRight': '20px'}),
        
        html.Div([
            html.Label('Выберите признак для анализа:', 
                      style={'fontWeight': 'bold', 'color': colors['text']}),
            dcc.Dropdown(
                id='feature-dropdown',
                options=[
                    {'label': 'Длина чашелистика', 'value': 'sepal_length'},
                    {'label': 'Ширина чашелистика', 'value': 'sepal_width'},
                    {'label': 'Длина лепестка', 'value': 'petal_length'},
                    {'label': 'Ширина лепестка', 'value': 'petal_width'},
                    {'label': 'Площадь чашелистика', 'value': 'sepal_area'},
                    {'label': 'Площадь лепестка', 'value': 'petal_area'},
                    {'label': 'Соотношение чашелистик/лепесток', 'value': 'sepal_petal_ratio'}
                ],
                value='petal_length',
                clearable=False,
                style={'width': '100%', 'marginBottom': '15px'}
            ),
        ], style={'width': '45%', 'display': 'inline-block'}),
    ], style={'marginBottom': '30px'}),
    
    html.Div([
        html.Div([
            dcc.Graph(id='pie-chart')
        ], style={'width': '33%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='bar-chart')
        ], style={'width': '33%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='box-chart')
        ], style={'width': '33%', 'display': 'inline-block'}),
    ]),
    
    html.Div([
        html.Div([
            dcc.Graph(id='scatter-chart')
        ], style={'width': '50%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Graph(id='histogram-chart')
        ], style={'width': '50%', 'display': 'inline-block'}),
    ]),
    
    html.Div([
        html.H3('Статистические показатели', style={'color': colors['header'], 'marginTop': '20px'}),
        html.Div(id='metrics-display', style={'display': 'flex', 'justifyContent': 'space-between', 'flexWrap': 'wrap'}),
    ]),
    
    html.Footer([
        html.Hr(),
        html.P('Дашборд создан в рамках практической работы по дисциплине "Базы данных и анализ промышленных данных"',
               style={'textAlign': 'center', 'color': colors['text'], 'fontSize': '12px'})
    ])
])

@app.callback(
    [Output('pie-chart', 'figure'),
     Output('bar-chart', 'figure'),
     Output('box-chart', 'figure'),
     Output('scatter-chart', 'figure'),
     Output('histogram-chart', 'figure'),
     Output('metrics-display', 'children')],
    [Input('species-dropdown', 'value'),
     Input('feature-dropdown', 'value')]
)
def update_dashboard(selected_species, selected_feature):
    filtered_df = df_iris.copy()
    
    if selected_species != 'ALL':
        filtered_df = filtered_df[filtered_df['species'] == selected_species]
    
    pie_fig = px.pie(filtered_df, names='species', 
                     title='Распределение по видам',
                     color='species',
                     color_discrete_map=species_colors)
    pie_fig.update_layout(paper_bgcolor=colors['background'])
    
    feature_names = {
        'sepal_length': 'Длина чашелистика (см)',
        'sepal_width': 'Ширина чашелистика (см)',
        'petal_length': 'Длина лепестка (см)',
        'petal_width': 'Ширина лепестка (см)',
        'sepal_area': 'Площадь чашелистика (см²)',
        'petal_area': 'Площадь лепестка (см²)',
        'sepal_petal_ratio': 'Соотношение чашелистик/лепесток'
    }
    
    bar_data = filtered_df.groupby('species')[selected_feature].mean().reset_index()
    bar_fig = px.bar(bar_data, x='species', y=selected_feature,
                     title=f'Среднее значение: {feature_names[selected_feature]}',
                     color='species',
                     color_discrete_map=species_colors)
    bar_fig.update_layout(paper_bgcolor=colors['background'], showlegend=False)
    
    box_fig = px.box(filtered_df, x='species', y=selected_feature,
                     title=f'Распределение: {feature_names[selected_feature]}',
                     color='species',
                     color_discrete_map=species_colors)
    box_fig.update_layout(paper_bgcolor=colors['background'], showlegend=False)
    
    scatter_fig = px.scatter(filtered_df, x='sepal_length', y='petal_length',
                             color='species', size='petal_width',
                             title='Зависимость длины лепестка от длины чашелистика',
                             color_discrete_map=species_colors,
                             labels={'sepal_length': 'Длина чашелистика (см)',
                                    'petal_length': 'Длина лепестка (см)'})
    scatter_fig.update_layout(paper_bgcolor=colors['background'])
    
    hist_fig = px.histogram(filtered_df, x=selected_feature, color='species',
                            nbins=20, opacity=0.7,
                            title=f'Распределение: {feature_names[selected_feature]}',
                            color_discrete_map=species_colors,
                            labels={selected_feature: feature_names[selected_feature]})
    hist_fig.update_layout(paper_bgcolor=colors['background'])
    
    total_samples = len(filtered_df)
    mean_value = filtered_df[selected_feature].mean()
    std_value = filtered_df[selected_feature].std()
    min_value = filtered_df[selected_feature].min()
    max_value = filtered_df[selected_feature].max()
    
    metrics = [
        html.Div([
            html.H4(f'{total_samples}', style={'color': colors['accent1'], 'fontSize': '28px'}),
            html.P('Объем выборки', style={'color': colors['text']})
        ], style={'textAlign': 'center', 'width': '18%', 
                  'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '10px'}),
        
        html.Div([
            html.H4(f'{mean_value:.2f}', style={'color': colors['accent2'], 'fontSize': '28px'}),
            html.P('Среднее значение', style={'color': colors['text']})
        ], style={'textAlign': 'center', 'width': '18%', 
                  'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '10px'}),
        
        html.Div([
            html.H4(f'{std_value:.2f}', style={'color': colors['accent3'], 'fontSize': '28px'}),
            html.P('Стандартное отклонение', style={'color': colors['text']})
        ], style={'textAlign': 'center', 'width': '18%', 
                  'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '10px'}),
        
        html.Div([
            html.H4(f'{min_value:.2f}', style={'color': colors['accent1'], 'fontSize': '28px'}),
            html.P('Минимум', style={'color': colors['text']})
        ], style={'textAlign': 'center', 'width': '18%', 
                  'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '10px'}),
        
        html.Div([
            html.H4(f'{max_value:.2f}', style={'color': colors['accent2'], 'fontSize': '28px'}),
            html.P('Максимум', style={'color': colors['text']})
        ], style={'textAlign': 'center', 'width': '18%', 
                  'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '10px'}),
    ]
    
    return pie_fig, bar_fig, box_fig, scatter_fig, hist_fig, metrics

if __name__ == '__main__':
    print("="*70)
    print("ЗАПУСК ДАШБОРДА")
    print("="*70)
    print("Дашборд доступен по адресу: http://127.0.0.1:8050")
    print("Для остановки сервера нажмите Ctrl+C")
    app.run(debug=True, host='127.0.0.1', port=8050)

ЗАПУСК ДАШБОРДА
Дашборд доступен по адресу: http://127.0.0.1:8050
Для остановки сервера нажмите Ctrl+C


Error on request:
Traceback (most recent call last):
  File "c:\Python314\Lib\site-packages\werkzeug\serving.py", line 370, in run_wsgi
    execute(self.server.app)
    ~~~~~~~^^^^^^^^^^^^^^^^^
  File "c:\Python314\Lib\site-packages\werkzeug\serving.py", line 355, in execute
    data = self.rfile.read(10_000_000)
MemoryError
Error on request:
Traceback (most recent call last):
  File "c:\Python314\Lib\site-packages\werkzeug\serving.py", line 370, in run_wsgi
    execute(self.server.app)
    ~~~~~~~^^^^^^^^^^^^^^^^^
  File "c:\Python314\Lib\site-packages\werkzeug\serving.py", line 355, in execute
    data = self.rfile.read(10_000_000)
MemoryError
Error on request:
Traceback (most recent call last):
  File "c:\Python314\Lib\site-packages\werkzeug\serving.py", line 370, in run_wsgi
    execute(self.server.app)
    ~~~~~~~^^^^^^^^^^^^^^^^^
  File "c:\Python314\Lib\site-packages\werkzeug\serving.py", line 355, in execute
    data = self.rfile.read(10_000_000)
MemoryError
